### **1. Price vs Quantity (basic relationship)**

In [0]:
%sql
SELECT
    Description,
    AVG(Price) AS avg_price,
    AVG(Quantity) AS avg_qty
FROM retail
GROUP BY Description
HAVING avg_price < 1000 AND avg_qty < 1000

-- removing outliers shows that products with price 0 to 80 have most demand at upto 400, this is like total elastic demand, however some outliers are also visible.

Description,avg_price,avg_qty
15CM CHRISTMAS GLASS BALL 20 LIGHTS,9.221524500907373,5.0
PINK CHERRY LIGHTS,7.283909774436093,8.575187969924812
WHITE CHERRY LIGHTS,7.111803278688529,8.30327868852459
"""RECORD FRAME 7"""" SINGLE SIZE """,3.5406932409012004,13.623916811091854
STRAWBERRY CERAMIC TRINKET BOX,1.5184548825710789,15.496497733827772
PINK DOUGHNUT TRINKET POT,2.141105047748988,9.155525238744884
SAVE THE PLANET MUG,1.7265291262135967,14.196601941747574
FANCY FONT HOME SWEET HOME DOORMAT,7.97682926829269,4.0426829268292686
CAT BOWL,2.9800000000000066,7.699186991869919
"DOG BOWL , CHASING BALL DESIGN",4.172205882352941,5.544117647058823


Databricks visualization. Run in Databricks to view.

### **2. Correlation overall**

In [0]:
from pyspark.sql.functions import corr

df = spark.read.table("retail")

df.groupBy("Description").agg(
    corr("Price", "Quantity").alias("corr")
).show(1000)

+--------------------+--------------------+
|         Description|                corr|
+--------------------+--------------------+
|15CM CHRISTMAS GL...|-0.28278995817688335|
|  PINK CHERRY LIGHTS| -0.3607828805116668|
| WHITE CHERRY LIGHTS|-0.39629805549321384|
|"RECORD FRAME 7""...| -0.4432492639287216|
|STRAWBERRY CERAMI...| -0.2180108408958959|
|PINK DOUGHNUT TRI...| -0.3094282782370328|
| SAVE THE PLANET MUG|-0.16279531171613512|
|FANCY FONT HOME S...| -0.3752778646368731|
|           CAT BOWL | -0.1154916346386237|
|DOG BOWL , CHASIN...|-0.11466539683803927|
|HEART MEASURING S...|                NULL|
|LUNCHBOX WITH CUT...| -0.2218401977018365|
|DOOR MAT BLACK FL...|-0.21964547073113663|
|LOVE BUILDING BLO...|-0.23238389163884157|
|HOME BUILDING BLO...|-0.23925748107019318|
|ASSORTED COLOUR B...|-0.03922356427735...|
| PEACE WOODEN BLO...|-0.18474445634616835|
|CHRISTMAS CRAFT W...|-0.18363252512174405|
|HEART IVORY TRELL...|-0.17436019819075668|
|HEART FILIGREE DO...| -0.188275

In [0]:
df.groupBy("Description").agg(
    corr("Price", "Quantity").alias("corr")
).filter("corr > 0").orderBy("corr", ascending=False).show(100)

+--------------------+-------------------+
|         Description|               corr|
+--------------------+-------------------+
|INCENSE BAZAAR CA...| 1.0000000000000002|
|BLUE PAINTED KASH...|                1.0|
|PINK PAINTED KASH...|                1.0|
| CAROUSEL PLACEMATS | 0.9999999999999999|
|BLACKCHRISTMAS TR...| 0.9827929550722722|
|PINK KASHMIRI COF...| 0.9790755624849468|
|PINK PAISLEY PHOT...| 0.8703882797784891|
|ACRYLIC HANGING J...|  0.806451612903226|
|FRESHWATER PEARL ...| 0.6981890746069254|
|COSY HOUR CIGAR B...| 0.6870237323092702|
|This is a test pr...| 0.6757731744600033|
|CAROUSEL CHILDREN...| 0.6024004577617609|
|TINY CRYSTAL BRAC...| 0.5879139922883987|
|  BEECH WOOD P/FRAME| 0.5416666666666669|
|CHERRY BLOSSOM PURSE| 0.5237229365663818|
|BOX OF 6 PEBBLE C...| 0.5207556439232954|
|DIAMANTE NECKLACE...| 0.5000000000000054|
|ROBOT PENCIL SHAR...|  0.500000000000001|
|NEW BAROQUE B'FLY...|0.49938887734343085|
|GREEN/BLUE CERAMI...|0.47519011407605316|
|ANTIQUE OL

In [0]:
from pyspark.sql.functions import corr, when, col

corr_df = df.groupBy("Description").agg(
    corr("Price", "Quantity").alias("corr")
)

bucketed = corr_df.withColumn(
    "category",
    when(col("corr") == 1, "perfect_positive")
    .when(col("corr") > 0.5, "strong_positive")
    .when((col("corr") > 0) & (col("corr") <= 0.5), "weak_positive")
    .when(col("corr") == 0, "no_relation")
    .when((col("corr") < 0) & (col("corr") >= -0.5), "weak_negative")
    .when(col("corr") < -0.5, "strong_negative")
)

bucketed.groupBy("category").count().show()

+----------------+-----+
|        category|count|
+----------------+-----+
|   weak_negative| 3648|
|            NULL|  740|
| strong_negative|  808|
|   weak_positive|  185|
| strong_positive|   16|
|perfect_positive|    2|
+----------------+-----+



### Correlation analysis indicates mixed price–quantity relationships across products. However, since correlation does not capture sensitivity, elasticity is computed to measure true demand response.

### **3. Elasticity Calculation**

In [0]:
%sql

-- using percentage method of elasticity calculation

WITH data AS (
    SELECT
        Description,
        DATE(InvoiceDate) AS dt,
        AVG(Price) AS price,
        SUM(Quantity) AS qty
    FROM retail
    GROUP BY Description, dt
),

calc AS (                         
    SELECT
        Description,
        dt,
        price,
        qty,
        LAG(price) OVER (PARTITION BY Description ORDER BY dt) AS prev_price,
        LAG(qty) OVER (PARTITION BY Description ORDER BY dt) AS prev_qty 
    FROM data
),

elasticity_row AS (
    SELECT
        Description,
        try_divide(
            try_divide(qty - prev_qty, prev_qty),
            try_divide(price - prev_price, prev_price)
        ) AS elasticity
    FROM calc
    WHERE 
        prev_price IS NOT NULL            
        AND prev_price != 0
        AND ABS((price - prev_price)/prev_price) > 0.1 
),

filtered AS (
    SELECT *
    FROM elasticity_row
    WHERE ABS(elasticity) < 5  
),

final AS (
    SELECT
        Description,
        AVG(elasticity) AS elasticity
    FROM filtered
    GROUP BY Description    --  final filter for grouping data per product
)

SELECT
    Description,      -- finally making case for elasticity categories and selecting 3 columns.
    elasticity,
    CASE
        WHEN elasticity > 0 THEN 'anomalous_positive'
        WHEN ABS(elasticity) > 1 THEN 'elastic'
        WHEN ABS(elasticity) = 1 THEN 'unitary'
        WHEN ABS(elasticity) < 1 THEN 'inelastic'
    END AS category
FROM final

Description,elasticity,category
DOORMAT UNION JACK GUNS AND ROSES,-0.059453416529136445,inelastic
3 STRIPEY MICE FELTCRAFT,-0.2584872594400989,inelastic
4 PURPLE FLOCK DINNER CANDLES,0.19262499551344167,anomalous_positive
50'S CHRISTMAS GIFT BAG LARGE,0.31831605373284355,anomalous_positive
BLACK PIRATE TREASURE CHEST,-1.452211152546965,elastic
BROWN PIRATE TREASURE CHEST,0.312209450358105,anomalous_positive
CAMPHOR WOOD PORTOBELLO MUSHROOM,-1.5270562770562772,elastic
CHERRY BLOSSOM DECORATIVE FLASK,0.6964285714285713,anomalous_positive
DOLLY GIRL BEAKER,-0.23643566248520928,inelastic
FLAMINGO LIGHTS,-0.7653111204356473,inelastic


Databricks visualization. Run in Databricks to view.

### The following conclusions are made based on elasticity outcomes:-
1. The anomalous positive means elasticity being completely positive, that means the Giffen or veblen goods whose demand seems to be increasing when price increases.

2. The elastic demand meant any elasticity greater then 1, but since positive are filtered out in CASE 1, then only negative will be in elastic demand cases, hereby stating the law of demand, the quantity changes in opposite direction of price change.

3. The inelastic demand meant that the elasticity is between 
-1 < inelastic < +1 . Example -0.3. This means that quantitiy demanded change is very less in terms of price change. That means people are willing to still pay for that good even if price is relatively higher.

4. Total 2725 inelastic goods, 591 elastic goods and 960 giffen goods are found in the data.

### **Note:- I compared segmentation robustness under |E|<10 and |E|<5. The stricter threshold reduced extreme-value bias and produced a more stable inelastic-dominant demand profile. Means at 5 performed well as compared to at 10, however only 82 units removed from list when value of elasticity taken 5 in place of 10.**

In [0]:
_sqldf.write.format("delta").mode("overwrite").saveAsTable("elasticity_table")

#saved the table as a delta

In [0]:
%sql
SELECT * FROM elasticity_table LIMIT 10

Description,elasticity,category
DOORMAT UNION JACK GUNS AND ROSES,-0.828,inelastic
3 STRIPEY MICE FELTCRAFT,0.0,inelastic
4 PURPLE FLOCK DINNER CANDLES,0.253,anomalous_positive
50'S CHRISTMAS GIFT BAG LARGE,-0.344,inelastic
BLACK PIRATE TREASURE CHEST,-1.965,elastic
BROWN PIRATE TREASURE CHEST,0.425,anomalous_positive
CAMPHOR WOOD PORTOBELLO MUSHROOM,-1.102,elastic
CHERRY BLOSSOM DECORATIVE FLASK,0.765,anomalous_positive
DOLLY GIRL BEAKER,-0.646,inelastic
FLAMINGO LIGHTS,-0.612,inelastic


This table is further used in next notebook named ADVANCED DEMAND ANALYSIS

In [0]:
%sql

WITH daily_product_data AS (
    SELECT
        Description,
        DATE(InvoiceDate) AS dt,
        AVG(Price) AS price,
        SUM(Quantity) AS qty
    FROM retail
    GROUP BY Description, dt
),

lagged_values AS (                --previous price and quantity is computed
    SELECT
        Description,
        price,
        qty,
        LAG(price) OVER (
            PARTITION BY Description
            ORDER BY dt
        ) AS prev_price,
        LAG(qty) OVER (
            PARTITION BY Description
            ORDER BY dt
        ) AS prev_qty
    FROM daily_product_data
),

daily_elasticity AS (
    SELECT
        Description,
        TRY_DIVIDE(
            TRY_DIVIDE(qty - prev_qty, prev_qty),
            TRY_DIVIDE(price - prev_price, prev_price)
        ) AS elasticity
    FROM lagged_values
    WHERE prev_price IS NOT NULL   -- to not take values that are null, zero or elasticity greater then 0.1 in abs terms so very small values filtered out.
      AND prev_price != 0        -- only calculate elasticity when price moved by more than 10%
      AND ABS(TRY_DIVIDE(price - prev_price, prev_price)) > 0.1
),

filterd_elasticity AS (
    SELECT
        Description,
        elasticity
    FROM daily_elasticity
    WHERE elasticity IS NOT NULL
      AND ABS(elasticity) < 5  -- elasticity again not to be taken above value 5, because extreme values with very small change in price and are inappropriate.
    -- The threshold was a robustness rule to remove mathematically unstable and business-unrealistic extremes. It was chosen empirically after observing that extreme values were driven by small denominator effects rather than true demand response.”
)

SELECT
    Description,
    ROUND(
        PERCENTILE_CONT(0.5)  -- Median logic is the middle value or 50th percentile 
        WITHIN GROUP (ORDER BY elasticity), 3  -- round truncate the large decimal values
    ) AS elasticity,     --- Final elasticity value computed using MEDIAN not mean cuz for Right Skewed Data, mean is pulled due to heavy extremes, and standard deviation was also more then MEDIAN method, so final elasticity value is the MEDIAN of daily elasiticity values per product with standard deviation computed other hand came 0.849 for MEDIAN method and 0.866 for mean method.
    CASE
        WHEN PERCENTILE_CONT(0.5)
             WITHIN GROUP (ORDER BY elasticity) > 0
            THEN 'anomalous_positive'
        WHEN ABS(
            PERCENTILE_CONT(0.5)
            WITHIN GROUP (ORDER BY elasticity)
        ) > 1
            THEN 'elastic'
        ELSE 'inelastic'
    END AS category
FROM filterd_elasticity
GROUP BY Description;

Description,elasticity,category
DOORMAT UNION JACK GUNS AND ROSES,-0.828,inelastic
3 STRIPEY MICE FELTCRAFT,0.0,inelastic
4 PURPLE FLOCK DINNER CANDLES,0.253,anomalous_positive
50'S CHRISTMAS GIFT BAG LARGE,-0.344,inelastic
BLACK PIRATE TREASURE CHEST,-1.965,elastic
BROWN PIRATE TREASURE CHEST,0.425,anomalous_positive
CAMPHOR WOOD PORTOBELLO MUSHROOM,-1.102,elastic
CHERRY BLOSSOM DECORATIVE FLASK,0.765,anomalous_positive
DOLLY GIRL BEAKER,-0.646,inelastic
FLAMINGO LIGHTS,-0.612,inelastic
